In [3]:
"""
AI Text Summarizer - Simple Extractive Summarization

Author: Kothapalli Madanmohan Reddy
Description:
    - Takes a long paragraph/article as input
    - Cleans and tokenizes text
    - Builds word frequency table
    - Scores sentences based on important words
    - Returns a shorter summary by selecting top-ranked sentences
"""

import re
from collections import Counter


# ---- Basic text utilities ---- #

def clean_text(text: str) -> str:
    """Normalize spaces and strip."""
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def sentence_tokenize(text: str):
    """
    Very simple sentence splitter based on punctuation.
    Not perfect, but works well enough for most plain text.
    """
    sentences = re.split(r"(?<=[.!?])\s+", text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences


def word_tokenize(text: str):
    """
    Lowercase, remove punctuation, split on whitespace.
    """
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    words = [w for w in text.split() if w]
    return words


def get_stopwords():
    """
    Minimal English stopword list to keep it dependency-free.
    You can expand this list if needed.
    """
    return {
        "the", "is", "are", "a", "an", "and", "or", "of", "to", "in",
        "on", "for", "with", "that", "this", "it", "as", "at", "by",
        "from", "be", "was", "were", "will", "would", "can", "could",
        "has", "have", "had", "do", "does", "did", "but", "if", "so",
        "not", "no", "about", "into", "than", "then", "too", "very"
    }


# ---- Core summarization logic ---- #

def build_frequency_table(words):
    stopwords = get_stopwords()
    filtered = [w for w in words if w not in stopwords]
    return Counter(filtered)


def score_sentences(sentences, freq_table):
    """
    Score each sentence by average frequency of its (non-stopword) words.
    """
    sentence_scores = {}
    for sent in sentences:
        words = word_tokenize(sent)
        if not words:
            continue

        score = 0
        useful_words = 0

        for w in words:
            if w in freq_table:
                score += freq_table[w]
                useful_words += 1

        if useful_words > 0:
            sentence_scores[sent] = score / useful_words

    return sentence_scores


def summarize(text: str, ratio: float = 0.3, max_sentences: int | None = None) -> str:
    """
    Create a summary of the given text.

    :param text: Input paragraph/article as string
    :param ratio: Fraction of sentences to keep (0–1). Default: 0.3 (30%)
    :param max_sentences: Optional hard cap on number of sentences.
    :return: Summary string
    """
    text = clean_text(text)
    sentences = sentence_tokenize(text)

    if not sentences:
        return ""

    words = word_tokenize(text)
    freq_table = build_frequency_table(words)
    sentence_scores = score_sentences(sentences, freq_table)

    # how many sentences to keep
    keep = max(1, int(len(sentences) * ratio))
    if max_sentences is not None:
        keep = min(keep, max_sentences)

    ranked = sorted(sentence_scores.items(), key=lambda x: x[1], reverse=True)
    selected = [s for s, _ in ranked[:keep]]

    # maintain original order
    ordered = [s for s in sentences if s in selected]
    return " ".join(ordered)


# ---- CLI entry point ---- #

def main():
    print("=== AI Text Summarizer ===")
    print("Paste your text below. End input with an empty line:\n")

    lines = []
    while True:
        try:
            line = input()
        except EOFError:
            break
        if line.strip() == "":
            break
        lines.append(line)

    full_text = "\n".join(lines)

    if not full_text.strip():
        print("No text provided. Exiting.")
        return

    # You can tune ratio/max_sentences here
    summary = summarize(full_text, ratio=0.35, max_sentences=5)

    print("\n--- Summary ---\n")
    print(summary)


if __name__ == "__main__":
    main()


=== AI Text Summarizer ===
Paste your text below. End input with an empty line:

Artificial Intelligence (AI) has rapidly evolved over the last decade and is now used in every industry. AI systems can learn from data, identify patterns, and make decisions with minimal human intervention. Machine learning, a subset of AI, allows computers to improve their performance using experience. Applications like voice assistants, recommendation systems, autonomous vehicles, and healthcare diagnosis are now part of our daily life. While AI offers huge benefits, it also includes challenges like ethical concerns, job impacts, and privacy risks.


--- Summary ---

AI systems can learn from data, identify patterns, and make decisions with minimal human intervention.
